# Blade Dance (Mercenary)

Blade Dance has two separate effects:

1. **Active.** A spin that hits every hostile in the 3×3 around the Mercenary for six Surround Hit
   strikes' worth of weapon damage, scaled by the Blade Dance skill. No hit roll and no dodge.
   It costs 100 endurance and has a 30 s cooldown.
2. **Passive amplifier.** In `do_attack`, once Blade Dance is *learned* (`skill[SK_BLADE_DANCE][0]
   != 0`), every Surround Hit strike deals **6×** damage. This ignores the Blade Dance skill level,
   the cooldown and endurance.

**Rebalanced.** The active's skill scale (shared with Deliver Death) used to be
`clamp(50 + p/2, 50, 150)`. It reached 150% at modified skill 200 and was flat after that, so
skill growth from 200 to the endgame (275–300) added nothing. The scale now uses the shared
mod-300 knee, `skill_knee_300(p, 50, 150)`: 150% at modified skill 300, then half the slope with
no hard limit. The active deals the same damage at 300 and more past it, but less below 300: at
the raise cap of 120 the scale drops from 110% to 90%. The passive 6× amplifier does not use the
scale and is unchanged. The "before" curves below show the old scale.

| | |
|---|---|
| Class | Mercenary |
| Target | Active: every hostile in the 3×3 around the caster (`aoe_base` 4) for players; NPCs use the 4-tile cross. Passive: Surround Hit's own targets |
| Cost | 100 endurance (active only) |
| Cooldown | 30 s (active only); started even if nothing is hit |
| Landing roll | Active: none. Passive: unchanged Surround rolls (`surround + rand(20) > fight skill`, then dodge) |
| Skill scale | Active only: `skill_knee_300(p, 50, 150)` %: 90% at 120, 150% at 300, 158% at 350; no hard limit |
| Damage | `do_hurt(sdam, 0)`: melee, reduced by armor, ¼ HP per point |
| Source | `server/src/driver/skill/mercenary.rs::skill_blade_dance`, `weapon_skill_scale_pct`; amplifier in `server/src/state/combat.rs::do_attack` |

---

# Formula

**Active.** With $W = \max(1, \text{weapon})$, the modified Blade Dance skill $p$ (negative counts
as 0) and the scale shared with Deliver Death:

$$
\text{base} = W + \text{rand}(W)
\qquad
\text{sdam} = \left\lfloor \frac{(\text{base} - \lfloor \text{base}/4 \rfloor)\cdot 6 \cdot \text{scale}}{100} \right\rfloor
$$

$$
\text{scale} = \begin{cases}
50 + \left\lfloor \dfrac{100\,p}{300} \right\rfloor & p \le 300\\[2ex]
150 + \left\lfloor \dfrac{100\,(p - 300)}{600} \right\rfloor & p > 300
\end{cases}\%
$$

Before: $\text{scale} = \operatorname{clamp}(50 + \lfloor p/2 \rfloor,\ 50,\ 150)$.

| Scale | Now | Before |
|---|---|---|
| At the raise caps (105 / 120) | 85% / 90% | 102% / 110% |
| 100% reached at | $p = 150$ | $p = 100$ |
| At the endgame band (275 / 300) | 141% / 150% | 150% / 150% |
| At $p = 350$ / 450 | 158% / 175% | 150% / 150% |
| Hard limit | none | 150% from $p = 200$ |

**Surround Hit with the amplifier.** Surround runs on swings that use the "surround" attack
animation, which is one of three animations and never repeats twice in a row. That makes it
**1/3 of swings** on average. It also needs the primary strike to land and not be dodged. Each
neighbour in the Surround footprint is then rolled separately:

$$
P_\text{roll} = P\big(s + \text{rand}(20) > \text{fight}(co)\big)
\qquad
\text{sdam}_\text{surround} = \big(\text{odam} - \lfloor \text{odam}/4 \rfloor\big)\times
\begin{cases} 6 & \text{Blade Dance learned} \\ 1 & \text{otherwise}\end{cases}
$$

Here `odam` is the primary strike's raw damage: $W + \text{rand}(6)+1 + \text{rand}(\lfloor
\text{STR}/2\rfloor)$, plus the extra dice on natural 1 and 2, before the hit-table bonus. The
Blade Dance skill scale does not apply.

| Surround base `skill[SK_SURROUND][0]` | Surround footprint (players) |
|---|---|
| 1 – 3 | 4-tile cross, only enemies attacking the Mercenary |
| 4 – 6 | 3×3 |
| 7 – 9 | 5×5 |
| ≥ 10 | 7×7 |

---

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import mag_model as mm

mm.apply_style()
P = mm.SKILL_RANGE
COOLDOWN_S = 30
END_COST = 100
SURROUND_SHARE = 1 / 3

# Model

In [ ]:
def scale_pct(p):
    """`weapon_skill_scale_pct`: `skill_knee_300(p, 50, 150)`."""
    return mm.knee_300(p, 50, 150)


def old_scale_pct(p):
    """Before the rebalance: `clamp(50 + p/2, 50, 150)`."""
    return mm.clamp(50 + mm.idiv(p, 2), 50, 150)


def active_rolls(weapon, p, scale=scale_pct):
    w = max(1, weapon)
    base = w + np.arange(w)
    return mm.idiv((base - mm.idiv(base, 4)) * 6 * scale(p), 100)


def active_expected_hp(weapon, p, armor=0, scale=scale_pct):
    """Expected HP removed per target by one active cast."""
    p = np.atleast_1d(p)
    out = np.array([np.mean(mm.hurt_hp(active_rolls(weapon, int(x), scale), armor)) for x in p])
    return mm._scalar(out if out.size > 1 else out[0])


def odam_pmf(weapon, strength, chance=12):
    """(values, probabilities) of `odam` given that the primary strike landed with `chance`."""
    normal = mm.melee_raw_rolls(weapon, strength)
    d6 = np.arange(6) + 1
    die2 = (normal[:, None] + d6[None, :]).ravel()
    two_d6 = (d6[:, None] + d6[None, :]).ravel()
    die1 = (normal[:, None] + two_d6[None, :]).ravel()
    parts = [(normal, (chance - 2) / chance), (die2, 1 / chance), (die1, 1 / chance)]
    vals = np.concatenate([v for v, _ in parts])
    probs = np.concatenate([np.full(len(v), w / len(v)) for v, w in parts])
    return vals, probs


def surround_strike_hp(weapon, strength, armor=0, blade_dance=True, chance=12):
    """Expected HP removed by one Surround strike that passes its roll."""
    vals, probs = odam_pmf(weapon, strength, chance)
    sdam = (vals - mm.idiv(vals, 4)) * (6 if blade_dance else 1)
    return float(np.sum(probs * mm.hurt_hp(sdam, armor)))


def surround_hp_per_window(surround, enemy_fight, weapon=40, strength=80, armor=0,
                           attacks_per_s=2.025, primary_hit=0.6, blade_dance=True,
                           window_s=COOLDOWN_S):
    """Expected Surround damage to one neighbour over `window_s` seconds of melee."""
    procs = window_s * attacks_per_s * SURROUND_SHARE * primary_hit
    p_roll = mm.p_rand20_exceeds(surround, enemy_fight)
    return procs * p_roll * surround_strike_hp(weapon, strength, armor, blade_dance)

# Validation

- `blade_dance_requires_endurance`: `a_end` 99,999 is refused.
- `blade_dance_spends_endurance_and_starts_cooldown`: 250,000 → 150,000, cooldown
  `active == TICKS*30`, and a second cast during the cooldown spends nothing.
- `weapon_skill_scale_reaches_one_and_a_half_at_skill_300` (shared with Deliver Death): 0 → 50,
  150 → 100, 300 → 150, 450 → 175.

In [ ]:
assert 99_999 < END_COST * 1000 and 250_000 - END_COST * 1000 == 150_000
assert mm.TICKS * COOLDOWN_S == 1080
assert [int(scale_pct(p)) for p in (0, 150, 300, 450)] == [50, 100, 150, 175]
assert [int(scale_pct(p)) for p in (105, 120, 275, 350)] == [85, 90, 141, 158]
assert [int(old_scale_pct(p)) for p in (0, 100, 120, 200, 400)] == [50, 100, 110, 150, 150]

# WV 40, skill 120: base 40..79, (base - base/4) * 6 * 0.90 (before * 1.10).
r = active_rolls(40, 120)
assert r.min() == 162 and r.max() == 324 and len(r) == 40
r = active_rolls(40, 120, old_scale_pct)
assert r.min() == 198 and r.max() == 396
# Same damage at 300 as the old ceiling.
assert active_expected_hp(40, 300) == active_expected_hp(40, 300, scale=old_scale_pct) == 101.25
vals, probs = odam_pmf(40, 80)
assert abs(probs.sum() - 1) < 1e-12 and vals.min() == 41
assert mm.attacks_per_second(mm.action_speed_row(18)) == 2.025
print("Blade Dance model OK")

# Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for wv in mm.WV_SWEEP:
    line, = ax.plot(P, active_expected_hp(wv, P), label=f"weapon value {wv}")
    if wv == 40:
        ax.plot(P, active_expected_hp(wv, P, scale=old_scale_pct), color=line.get_color(),
                lw=1.2, ls=":", label="before, weapon value 40")
mm.skill_axis(ax, "expected HP per target (armor 0)", "Active: damage per target per cast")
mm.mark_caps(ax, extra=[(200, "old 150% at 200"), (300, "150% at 300")])
ax.legend(loc="upper left", bbox_to_anchor=(0.0, 0.78), fontsize=8)

ax = axes[1]
for armor in mm.ARMOR_SWEEP:
    line, = ax.plot(P, active_expected_hp(40, P, armor), label=f"armor {armor}")
    if armor == 0:
        ax.plot(P, active_expected_hp(40, P, armor, scale=old_scale_pct), color=line.get_color(),
                lw=1.2, ls=":", label="before, armor 0")
mm.skill_axis(ax, "expected HP per target", "Active vs armor (weapon value 40)")
mm.mark_caps(ax)
ax.legend(loc="upper left", bbox_to_anchor=(0.0, 0.88), fontsize=8)
plt.tight_layout()

In [ ]:
# Passive amplifier: one Surround strike vs one active hit, per target.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
wvs = np.array(mm.WV_SWEEP)
plain = [surround_strike_hp(w, 80, blade_dance=False) for w in wvs]
amped = [surround_strike_hp(w, 80, blade_dance=True) for w in wvs]
active_mid = [active_expected_hp(w, 120) for w in wvs]
active_end = [active_expected_hp(w, 300) for w in wvs]
x = np.arange(len(wvs))
bw = 0.2
ax.bar(x - 1.5 * bw, plain, width=bw, label="Surround strike, no Blade Dance")
ax.bar(x - 0.5 * bw, amped, width=bw, label="Surround strike, Blade Dance learned")
ax.bar(x + 0.5 * bw, active_mid, width=bw, label="Blade Dance active (skill 120)")
ax.bar(x + 1.5 * bw, active_end, width=bw, label="Blade Dance active (skill 300)")
ax.set_xticks(x, [f"WV {w}" for w in wvs])
ax.set_ylabel("expected HP per target (STR 80, armor 0)")
ax.set_title("One Surround strike vs one Blade Dance hit")
ax.legend(fontsize=8)

# Over one 30 s cooldown window, x = Surround modified skill vs neighbours with fight skill 100.
ax = axes[1]
ax.plot(P, surround_hp_per_window(P, 100, blade_dance=True), label="Surround passive, Blade Dance learned")
ax.plot(P, surround_hp_per_window(P, 100, blade_dance=False), label="Surround passive, no Blade Dance")
for p, ls in ((120, "--"), (300, "-.")):
    ax.axhline(active_expected_hp(40, p), color="tab:red", ls=ls, lw=1.5,
               label=f"one Blade Dance active, skill {p}")
ax.axhline(active_expected_hp(40, 120, scale=old_scale_pct), color="tab:red", ls=":", lw=1.2,
           label="before, one active at skill 120")
mm.skill_axis(ax, "expected HP per neighbour per 30 s", "Per 30 s: passive amplifier vs the active (WV 40)")
ax.set_xlabel("Surround Hit modified skill (neighbour fight skill 100)")
mm.mark_caps(ax)
ax.legend(loc="center right", fontsize=8)
plt.tight_layout()

The right-hand chart assumes the Mercenary swings 2.025 times per second (base speed 18) and
lands 60% of primary strikes (equal Weapon Skill). Surround then fires on about 12 swings per 30 s.

# Observations

- **The passive amplifier is still the real ability, and it is very strong.** The rebalance does
  not touch it. Learning Blade Dance at any level makes every Surround strike deal 6×. At WV 40,
  STR 80 a Surround strike goes from 12 to 72 HP per neighbour. That is more than four normal
  landed hits (15.75 HP each) on *every* neighbour, from one swing in three.
- **Over a 30 s window the passive does 8.7× to 14.5× the active.** At 2.025 swings/s and 60% to
  hit, Surround fires about 12 times per 30 s. Against neighbours with fight skill 100, a Surround
  skill of 105 deals 880 HP per neighbour in that window. One active cast deals 61 HP at skill
  120 (14.5×; before: 74, 11.9×) and 101 HP at skill 300 (8.7×). Without Blade Dance, the same
  Surround deals 147.
- **The active is weaker mid-game and the same at the endgame.** At WV 40 it deals 57 / 61 HP per
  target at skill 105 / 120 (before: 69 / 74, −18% at 120). At the endgame band it deals 95 HP at
  275 (before: 101) and 101 HP at 300, the same as the old ceiling. At 350 it deals 107 HP, +5.2%
  over 300, where before skill past 200 added nothing.
- **The amplifier ignores the Blade Dance skill.** It checks only `skill[SK_BLADE_DANCE][0] != 0`.
  One point in Blade Dance gives the full 6×. Further raises only improve the active, and now do
  so all the way to 300 and beyond (scale 85% at 105, 90% at 120, 150% at 300). The lower
  mid-game active makes the gap to the passive wider at the raise cap.
- **The amplified Surround also gets the largest footprint.** It uses the Surround base: 7×7
  (48 tiles) at base ≥ 10, with no "must be attacking me" check outside the legacy cross. The
  active always covers the 3×3 (8 tiles) for players.
- **The active is an endurance-gated AoE with no roll.** Armor is subtracted once from a large
  hit, so armor 40 costs 16% at skill 120 (61 → 51) and 10% at 300 (101 → 91). NPC casters get
  the 4-tile cross, without the attacking check Surround uses.
- **Balance suggestions (still open).** The 6× amplifier is not addressed by the mod-300 knee.
  Gate the 6× on a short buff that the active applies (for example "Surround strikes ×3 for
  10 s"). Or drop the amplifier and keep the active as the payoff. As it stands, the active
  barely matters next to the passive it unlocks, and the skill scale now only affects the part of
  the ability that matters least.